In [ ]:
#!/usr/bin/env python3
import os
import time
import json
import argparse
import random
from typing import Any, Optional
import requests


# ----------------------------
# Helpers
# ----------------------------
def build_headers(api_key: str) -> dict[str, str]:
    return {"X-API-Key": api_key, "Content-Type": "application/json"}

def _raise_for_api_error(resp: requests.Response) -> None:
    if 200 <= resp.status_code < 300:
        return
    try:
        data = resp.json()
        detail = data.get("detail") if isinstance(data, dict) else None
    except Exception:
        detail = None
    msg = f"HTTP {resp.status_code}"
    if detail:
        msg += f": {detail}"
    raise RuntimeError(msg)


def api_get(base_url: str, path: str, api_key: str, params: Optional[dict[str, Any]] = None) -> Any:
    url = f"{base_url}{path}"
    resp = requests.get(url, headers=build_headers(api_key), params=params, timeout=15)
    _raise_for_api_error(resp)
    return resp.json()


def api_post(base_url: str, path: str, api_key: str, body: dict[str, Any]) -> Any:
    url = f"{base_url}{path}"
    resp = requests.post(url, headers=build_headers(api_key), data=json.dumps(body), timeout=10)
    if not (200 <= resp.status_code < 300):
        try:
            err = resp.json().get("detail", "")
        except Exception:
            err = resp.text
        raise RuntimeError(f"HTTP {resp.status_code}: {err}")
    return resp.json()


def place_order(api_url: str, api_key: str, order: dict[str, Any]) -> None:
    """Send a single order to the API."""
    try:
        res = api_post(api_url, "/api/v1/orders", api_key, order)
        print(
            f"[OK] {order['side'].upper():4} {order['symbol']:5} "
            f"{order['quantity']:>3} @ {order.get('price', 'MKT')} ({order['order_type']})"
        )
    except Exception as e:
        print(f"[ERR] Failed order for {order['symbol']}: {e}")

def list_symbols(base_url: str, api_key: str) -> None:
    data = api_get(base_url, "/api/v1/symbols", api_key)
    symbols = data.get("symbols", [])
    if not symbols:
        print("No symbols available.")
        return
    print("Available symbols:")
    for row in symbols:
        print(f"  - {row.get('symbol')}\t{row.get('name')}")

def list_open_orders(base_url: str, api_key: str, symbol: Optional[str] = None) -> None:
    params: dict[str, Any] = {}
    if symbol:
        params["symbol"] = symbol
    data = api_get(base_url, "/api/v1/orders/open", api_key, params=params)
    orders = data.get("orders", [])
    if not orders:
        print("No open orders.")
        return
    print("Open orders:")
    for o in orders:
        price_str = f" @ {o['price']}" if o.get("price") is not None else ""
        print(
            f"  - {o['order_id']} | {o['symbol']} {o['side']} {o['quantity']} {o['order_type']}{price_str} | {o['status']}"
        )


# ----------------------------
# Market-making logic
# ----------------------------
def fetch_quote(base_url: str, api_key: str, symbol: str) -> dict:
    bid = float(q.get("bid") or q.get("best_bod) or 0.0"))
    ask = float(q.get("ask") or q.get("best_aod) or 0.0"))
    last = float(q.get("last") or q.get("price") or 0.0)
    return {"bid": bid, "ask": ask, "last": last}
 
def generate_fair_values(api_url: str, api_key: str, symbols: list[str]) -> dict[str, float]:
    fair = {}
    for s in symbols:
        q = fetch_quote(api_url, api_key, s)
        if q["bid"] and q["ask"]:
            fair[s] = round((q["bid"] + q["ask"]) / 2.0, 2)
        elif q["last"]:
            fair[s] = round(q["last"], 2)
        else:
            fair[s] = 50.00    
    return fair


def update_fair_values(current_fair: float, mid: float, alpha: float = 0.1) -> float:
    if mid <=0:
        return current_fair
    return round((1-alpha) * current_fair + alpha * mid, 2)


def make_bid_ask_orders_from_quote(symbol: str,
                                fair_value: float,
                                bid: float,
                                ask: float,
                                min_qty: int = 1,
                                max_qty: int = 5,
                                edge_cents: float = 0.03,
                                tick: float = 0.01) -> list[dict[str, any]]:
    if bid <= 0 or ask <= 0 or ask <= bid:
        return []
        
    half_spread = max((ask - bid) / 2.0, tick)
    buy_px  = min(fair_value - edge_cents, (bid + ask) / 2.0 - half_spread + tick)
    sell_px = max(fair_value + edge_cents, (bid + ask) / 2.0 + half_spread - tick)
    
    buy_px = min(buy_px, bid
    sell_px = max(sell_px, ask)
    
    buy_px = round(max(tick, buy_px), 2)
    sell_px = round(max(tick, sell_px + tick), 2)
    
    import random
    qty_buy = random.randint(min_qty, max_qty)
    qty_sell = random.randint(min_qty, max_qty)

    return [
        {"symbol": symbol, "side": "buy", "order_type": "limit", "quantity": qty, "price": bid_px},
        {"symbol": symbol, "side": "sell", "order_type": "limit", "quantity": qty, "price": ask_px},
    ]


# ----------------------------
# Main trading loop
# ----------------------------
def market_making_loop(api_url: str, api_key: str, symbols: list[str], loop: bool = True):
    import time
fair = generate_fair_values(api_url, api_key, symbols)
print("Initial fair values:", fair)

MIN_SPREAD = 0.02
MAX_SPREAD = 1.5
SLEEP_SEC = 0.5

while True:
    try:
        for sym in symbols: 
            q = fetch_quote(api_url, api_key, sym)
                bid, ask = q["bid"], q["ask"]
                if not bid or not ask:
                    continue
                spread = ask - bid
                mid = (bid + ask) / 2.0

                # update fair toward mid (EMA)
                fair[sym] = update_fair_value_ema(fair[sym], mid, alpha=0.2)

                # only quote inside a reasonable spread window
                if not (MIN_SPREAD <= spread <= MAX_SPREAD):
                    continue

                orders = make_bid_ask_orders_from_quote(
                    symbol=sym,
                    fair_value=fair[sym],
                    bid=bid,
                    ask=ask,
                    min_qty=1,
                    max_qty=3,
                    edge_cents=0.03,   # widen to 0.05–0.08 if you get picked off
                    tick=0.01,
                )

                for o in orders:
                    place_order(api_url, api_key, o)
                    print(f"[{sym}] {o['side'].upper()} {o['quantity']} @ {o['price']:.2f} "
                        f"(bid {bid:.2f} / ask {ask:.2f} / fair {fair[sym]:.2f})")

                time.sleep(SLEEP_SEC)

            if not loop:
                break

        except KeyboardInterrupt:
            print("\nStopping market-making loop.")
            break
        except Exception as e:
            print("Loop error:", e)
            time.sleep(1)

# ----------------------------
# Entry point
# ----------------------------
def parse_args():
    parser = argparse.ArgumentParser(description="Automated Market Maker for CTC API")
    parser.add_argument("--api-url", default=os.environ.get("CTC_API_URL", "http://localhost:8000"))
    parser.add_argument("--api-key", default=os.environ.get("CTC_API_KEY") or os.environ.get("X_API_KEY"))
    parser.add_argument("--symbols", default="AAA,BBB,CCC,ETF", help="Comma-separated list of symbols")
    parser.add_argument("--loop", action="store_true", help="Continuously place orders")
    return parser.parse_args()


def main():
    args = parse_args()
    api_key = args.api_key or input("Enter API key: ").strip()
    if not api_key:
        print("API key required.")
        return 1

    symbols = [s.strip().upper() for s in args.symbols.split(",") if s.strip()]
    market_making_loop(args.api_url, api_key, symbols, loop=args.loop)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())